In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.4 Dataset splits

> 🎯 **Goal:** Split the data into a training portion and a held-out validation portion, and understand why measuring on data the model never saw is the only honest test.

**Why this matters.** If you only ever measure the model on the exact text it trained on, you learn nothing about whether it actually understands the language or just memorized those specific passages. The validation set (a slice of data you set aside and never train on) is your honest yardstick. Comparing training loss against validation loss is how every later diagnosis, overfitting, early stopping, model selection, becomes possible. Without a clean split you are flying blind.

**The intuition.** Studying for an exam from a textbook is training. The closed-book exam is validation. If the exam reuses the exact practice questions you already memorized, a high score proves nothing, you might have learned the material or just memorized those answers. The exam is only meaningful when its questions are ones you did not study. Letting exam questions leak into your study material (data leakage) is cheating, and it makes a weak student look brilliant right up until it matters.

**The mechanics.** `split_data(data, val_frac=0.1)` carves the token stream into two contiguous pieces: the first 90% becomes `train` and the last 10% becomes `val`. Contiguous matters for a language model, because the data is a sequence and we sample windows from it. If we shuffled tokens before splitting, windows straddling the cut would share neighboring text across both sets, quietly leaking the validation content into training. Keeping each split as one unbroken run, with no overlap, keeps the exam sealed. The printout confirms the two sizes add up exactly to the original, proof nothing was duplicated or dropped.

In [ ]:
from pocketlm import CharTokenizer
corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)

from pocketlm import split_data
train, val = split_data(data, val_frac=0.1)
print("train tokens:", len(train), " val tokens:", len(val))
print("contiguous, no overlap:", len(train) + len(val) == len(data))

**What just happened.** The printout reports the token counts for each split and confirms `len(train) + len(val) == len(data)`, every token accounted for, none shared. You now hold a sealed exam: a tenth of the corpus the model will never train on, ready to give you honest readings in the lessons ahead.

⚠️ **Common pitfalls**
- Shuffling tokens before splitting a sequence model's data: windows near the cut leak text across the boundary, inflating validation scores.
- Touching the validation set during training (using it to pick batches, tune weights, or anything beyond pure measurement): the moment you train on it, it stops being a fair exam.
- A validation set too tiny to be stable: with only a handful of tokens the val loss is noisy and its swings tell you nothing reliable.

🏋️ **Try it yourself.** Change `val_frac` to 0.2 and rerun: the validation set doubles and the assertion's target shifts to 0.2. Then try a tiny `val_frac=0.001` and reason about why such a small exam would give you a jumpy, untrustworthy validation loss in the next lessons.

In [ ]:
assert len(train) + len(val) == len(data)
assert abs(len(val) / len(data) - 0.1) < 0.01